# Graphing Runnables

In [1]:
# run the line of code below to check the version of langchain in the current environment.
# substitute "langchain" with any other package name to check their version.

In [50]:
pip show langchain

Name: langchain
Version: 1.3.1
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\tools\anaconda3\envs\langchain_env\lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
%load_ext dotenv
%dotenv

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [5]:
chat_template_tools = ChatPromptTemplate.from_template('''
What are the five most important tools a {job title} needs?
Answer only by listing the tools.
''')

chat_template_strategy = ChatPromptTemplate.from_template('''
Considering the tools provided, develop a strategy for effectively learning and mastering them:
{tools}
''')

In [6]:
chat = ChatOpenAI(model_name = 'gpt-4', 
                  # model_kwargs = {'seed':365},
                  seed = 365,
                  temperature = 0,
                  max_tokens = 100)

In [7]:
string_parser = StrOutputParser()

In [8]:
# Use of Passthrough
chain_long = (chat_template_tools | chat | string_parser | {'tools':RunnablePassthrough()} | 
              chat_template_strategy | chat | string_parser)

In [9]:
chain_long.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
     +-------------+       
     | Passthrough |       
     +-------------+       
            *              
            *              
            *       

# RunnableParallel

In [10]:
from langchain_core.runnables import RunnableParallel

In [11]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books. 
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers. 
    Answer only by listing the projects.
    '''
)

In [12]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [13]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [14]:
chain_parallel.invoke({'programming language':'Python'})

{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Web Scraper using BeautifulSoup.\n2. Developing a simple Machine Learning model using Scikit-learn.\n3. Creating a basic web application using Django or Flask.'}

In [15]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

In [16]:
%%time
chain_books.invoke({'programming language':'Python'})

CPU times: total: 0 ns
Wall time: 1.89 s


'1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin'

In [17]:
%%time
chain_projects.invoke({'programming language':'Python'})

CPU times: total: 0 ns
Wall time: 1.25 s


'1. Building a Web Scraper using BeautifulSoup and Requests\n2. Developing a simple Machine Learning application with Scikit-learn\n3. Creating a GUI application with Tkinter or PyQt'

In [18]:
%%time
# keep in mind the output is dictionary
chain_parallel.invoke({'programming language':'Python'})

CPU times: total: 0 ns
Wall time: 2.61 s


{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Web Scraper using BeautifulSoup.\n2. Developing a simple Machine Learning model using Scikit-learn.\n3. Creating a GUI application with Tkinter.'}

In [19]:
# if you run the above three, you will reach to the conclusion that running in parallel is more time efficient

# Piping a RunnableParallel with Other Runnables

In [20]:
# add a new variable. Expecting the output from books and projects. We require the completion of expected time.
chat_template_time = ChatPromptTemplate.from_template(
     '''
     I'm an intermediate level programmer.
     
     Consider the following literature:
     {books}
     
     Also, consider the following projects:
     {projects}
     
     Roughly how much time would it take me to complete the literature and the projects?
     
     '''
)

In [21]:
# increase the tokens to 500
chat = ChatOpenAI(model_name = 'gpt-4', 
                  # model_kwargs = {'seed':365},
                  seed = 365,
                  temperature = 0,
                  max_tokens = 500)

In [22]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [23]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [24]:
chain_parallel.invoke({'programming language':'Python'})

{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Web Scraper using BeautifulSoup and Requests\n2. Developing a simple Machine Learning application with Scikit-learn\n3. Creating a GUI application with Tkinter or PyQt'}

In [25]:
# construct a chain and feed this as an input as the chain_time variable

In [26]:
chain_time1 = (RunnableParallel({'books':chain_books, 
                                'projects':chain_projects}) 
              | chat_template_time 
              | chat 
              | string_parser
             )

In [27]:
# removed the RunnableParallel wrapper because chat_template_time (other Runnable) will automatically take care of it. 
chain_time2 = ({'books':chain_books, 
                'projects':chain_projects}
              | chat_template_time 
              | chat 
              | string_parser
             )

In [28]:
print(chain_time2.invoke({'programming language':'Python'}))

The time it takes to complete the literature and the projects can vary greatly depending on several factors such as your current skill level, the amount of time you can dedicate each day, your reading speed, and how quickly you grasp new concepts. 

However, as a rough estimate:

1. "Fluent Python: Clear, Concise, and Effective Programming" - This book is around 800 pages. If you read and practice for about 2 hours a day, it might take you around 1-2 months to complete.

2. "Python Cookbook: Recipes for Mastering Python 3" - This book is around 700 pages. Again, if you read and practice for about 2 hours a day, it might take you around 1-2 months to complete.

3. "Effective Python: 90 Specific Ways to Write Better Python" - This book is around 230 pages. If you read and practice for about 2 hours a day, it might take you around 2-3 weeks to complete.

As for the projects:

1. Building a Web Scraper using BeautifulSoup - If you're familiar with the basics of web scraping, this project m

In [29]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

# RunnableLambda

In [30]:
# RunnableLambda lets you wrap any Python function and use it as a step inside an LCEL chain.
# That means you can take any Python logic — simple or complex — and plug it directly into your LLM pipeline.
# RunnableLambda is used whenever you need to do something that isn’t an LLM call but still belongs inside your chain.
# RunnableLambda is how you inject custom Python logic into an LCEL chain — preprocessing, postprocessing, routing, or 
# integrating external tools — making your pipelines flexible and production‑ready
from langchain_core.runnables import RunnableLambda

In [31]:
find_sum = lambda x: sum(x)

In [32]:
find_sum([1, 2, 5])

8

In [33]:
find_square = lambda x: x**2

In [34]:
find_square(8)

64

In [35]:
runnable_sum = RunnableLambda(lambda x: sum(x))

In [36]:
runnable_sum.invoke([1, 2, 5])

8

In [37]:
runnable_square = RunnableLambda(lambda x: x**2)

In [38]:
runnable_square.invoke(8)

64

In [39]:
chain = runnable_sum | runnable_square

In [40]:
chain.invoke([1, 2, 5])

64

In [41]:
chain.get_graph().print_ascii()

+-------------+  
| LambdaInput |  
+-------------+  
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
+--------------+ 
| LambdaOutput | 
+--------------+ 


# The @chain Decorator

In [42]:
# The  decorator runs because it converts your function into a LangChain Runnable, giving 
# it LCEL execution behavior instead of normal Python function behavior.
from langchain_core.runnables import chain

In [43]:
def find_sum(x):
    return sum(x)

def find_square(x):
    return x**2

In [44]:
chain1 = RunnableLambda(find_sum) | RunnableLambda(find_square)

In [45]:
chain1.invoke([1, 2, 5])

64

In [46]:
@chain
def runnable_sum(x):
    return sum(x)

@chain
def runnable_square(x):
    return x**2

In [47]:
type(runnable_sum), type(runnable_square)

(langchain_core.runnables.base.RunnableLambda,
 langchain_core.runnables.base.RunnableLambda)

In [48]:
chain2 = runnable_sum | runnable_square

In [49]:
chain2.invoke([1, 2, 5])

64